# Distribution analysis, rectification check & PADS cross-dataset

Covers the analyses that explain **why** PD-vs-ET is hard and what does/doesn't help:

1. **Frequency distributions** (max / mean / median) by class + overlap → the root cause
2. **Rectification check** — magnitude vs PC1, and why we keep magnitude
3. **PADS cross-dataset** — transfer, pooling, domain shift, augmentation

Run top-to-bottom. Sections 3+ need `pads_stretchhold/` (see `pdetn/README_PADS.md`).

## 0. Setup

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == "pdetn": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from collections import defaultdict
from scipy.signal import welch, butter, sosfiltfilt
from scipy.stats import kruskal, mannwhitneyu
from sklearn.model_selection import GroupKFold
from tremor.data import CLASS_NAMES
from tremor.stats import bootstrap_subject_ci
from pdetn.crossdataset import load_local_sensor, load_pads_extracted, build_features, dataset_identity_probe
from pdetn.model import TwoStageClassifier
from pdetn.evaluate import evaluate
from pdetn.deep_crossdataset import pd_vs_et_metrics
LO, HI, FS = 3.0, 15.0, 100.0
CLASS_COLORS = {"N":"#2c7fb8","PD":"#d95f02","ET":"#1b9e77"}
DATA_ROOT, PADS_DIR = "Data", "pads_stretchhold"
print("ready")

## 1. Frequency distributions by class

Plain measures first, before any fancy transform:
* **max (peak)** frequency — argmax of the PSD in 3–15 Hz
* **mean** frequency — power-weighted centroid
* **median** frequency — 50% cumulative power\n\nDefault `mode='perchannel'` computes PSDs per axis (no rectification — see §2).

In [ ]:
def freq_measures(x, mode="perchannel"):
    """Return (peak, mean, median) frequency in the tremor band.
    mode='perchannel' -> PSD per channel then summed (NO rectification)
    mode='magnitude'  -> bandpass then vector magnitude (RECTIFIES: doubles freq)"""
    if mode == "magnitude":
        sos = butter(4,[LO,HI],'band',fs=FS,output='sos')
        sig = np.sqrt((sosfiltfilt(sos,x,axis=-1)**2).sum(0))
        n = int(min(256,len(sig))); f,P = welch(sig,fs=FS,nperseg=n)
    else:
        n = int(min(256,x.shape[1])); f,P = welch(x,fs=FS,nperseg=n,axis=-1); P = P.sum(0)
    m = (f>=LO)&(f<HI); f,P = f[m], P[m]+1e-18
    peak = float(f[np.argmax(P)])
    mean = float((f*P).sum()/P.sum())
    c = np.cumsum(P)/P.sum(); med = float(f[np.searchsorted(c,0.5)])
    return peak, mean, med

def distribution_table(recs, tag, mode="perchannel"):
    rows = {c:{'peak':[], 'mean':[], 'med':[]} for c in CLASS_NAMES}
    for r in recs:
        p,mn,md_ = freq_measures(r.x, mode); c = CLASS_NAMES[r.y]
        rows[c]['peak'].append(p); rows[c]['mean'].append(mn); rows[c]['med'].append(md_)
    print(f"\n===== {tag}  (mode={mode}) =====")
    print(f"{'measure':>8} {'class':>4} {'median':>7} {'IQR':>15} {'n':>5}")
    for meas in ['peak','mean','med']:
        for c in CLASS_NAMES:
            v = np.array(rows[c][meas]); q1,q3 = np.percentile(v,[25,75])
            print(f"{meas:>8} {c:>4} {np.median(v):>7.2f}  [{q1:5.2f},{q3:5.2f}] {len(v):>5}")
        g = [np.array(rows[c][meas]) for c in CLASS_NAMES]
        pe = mannwhitneyu(rows['PD'][meas], rows['ET'][meas])
        eff = 2*pe.statistic/(len(rows['PD'][meas])*len(rows['ET'][meas]))-1
        print(f"{'':>8} -> KW(3-class) p={kruskal(*g).pvalue:.2e}   "
              f"PD-vs-ET p={pe.pvalue:.4f} effect={eff:+.2f}")
    return rows

local = load_local_sensor(DATA_ROOT, action="OUT", sensor="lower_arm")
L = distribution_table(local, "LOCAL (lower_arm, OUT)")

### Distribution overlap — the key number\n`1.0` = identical distributions, `0` = disjoint.

In [ ]:
def overlap(a, b, bins=30):
    lo, hi = min(min(a),min(b)), max(max(a),max(b))
    ha,_ = np.histogram(a,bins=bins,range=(lo,hi),density=True)
    hb,_ = np.histogram(b,bins=bins,range=(lo,hi),density=True)
    return float(np.minimum(ha,hb).sum()*(hi-lo)/bins)

print("PD-vs-ET distribution overlap (LOCAL):")
for meas in ['peak','mean','med']:
    print(f"  {meas:>5}: {overlap(L['PD'][meas], L['ET'][meas]):.2f}")
print("\n-> roughly half to two-thirds of the PD and ET distributions coincide, and the")
print("   PD-vs-ET tests above are NOT significant (p ~ 0.25-0.47), while N separates")
print("   at p<1e-10. The classes are not separated along the frequency axis --")
print("   which is why no time-frequency method improved PD-vs-ET.")

### Visualise the distributions

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(14,4))
for ax, meas, title in zip(axes, ['peak','mean','med'],
                           ['max (peak) frequency','mean frequency','median frequency']):
    for c in CLASS_NAMES:
        ax.hist(L[c][meas], bins=20, range=(LO,HI), alpha=0.5,
                label=f"{c} (n={len(L[c][meas])})", color=CLASS_COLORS[c], density=True)
    ax.set_title(title); ax.set_xlabel("Hz"); ax.legend(fontsize=8)
axes[0].set_ylabel("density")
plt.suptitle("LOCAL lower_arm/OUT — N separates, PD & ET overlap"); plt.tight_layout(); plt.show()

## 2. Rectification check: magnitude vs PC1

`sqrt(gx²+gy²+gz²)` **squares** the signal, so a 6 Hz tremor shows up at 12 Hz.
`pdetn/signal_features.py` and `spatial_features.py` use this magnitude.

In [ ]:
print(f"{'class':>5} {'peak MAGNITUDE':>16} {'peak PER-CHANNEL':>18} {'ratio':>7}")
for c in CLASS_NAMES:
    sub = [r for r in local if CLASS_NAMES[r.y]==c]
    m  = np.median([freq_measures(r.x,"magnitude")[0]  for r in sub])
    ch = np.median([freq_measures(r.x,"perchannel")[0] for r in sub])
    print(f"{c:>5} {m:>16.2f} {ch:>18.2f} {m/ch:>7.2f}")
print("\n-> ratio ~2 for ET confirms rectification doubles the apparent frequency.")
print("-> per-channel: PD and ET share the SAME median peak (6.64 Hz) on lower_arm.")

### Does removing the rectification help the classifier?
`mode='pc1'` projects onto the first principal component (dominant oscillation
axis) — signed, no rectification.

In [ ]:
from pdetn.signal_features import advanced_features, ADVANCED_FEATURE_NAMES
from pdetn.spatial_features import spatial_features, SPATIAL_FEATURE_NAMES
from pdetn.separability import patient_decomp_features
from tremor.quaternion_data import load_quaternion_recordings

recs9 = load_quaternion_recordings(DATA_ROOT, action="OUT", mode="angular_velocity")  # all 9 ch
Xtf, y, subj = patient_decomp_features(recs9,"stft",nperseg=256,nfft=256,noverlap=192)

def patient_feats(fn, names, mode):
    per = defaultdict(list)
    for r in recs9:
        d = fn(r.x, mode=mode); per[r.subject].append([d[f] for f in names])
    p = sorted(per); return np.array([np.nanmean(per[k],0) for k in p]), np.array(p)

print(f"{'features':>26}{'macroF1':>9}{'ET_F1':>8}{'PDvsET':>8}")
for mode in ["magnitude","pc1"]:
    Xsp,_ = patient_feats(spatial_features, SPATIAL_FEATURE_NAMES, mode)
    for tag, X in [(f"spatial [{mode}]", Xsp),
                   (f"TF+spatial [{mode}]", np.concatenate([Xtf,Xsp],1))]:
        r = evaluate(lambda: TwoStageClassifier("logreg","logreg",tune_et_threshold=True),
                     X, y, subj, n_boot=500)
        print(f"{tag:>26}{r['macro_f1']:>9.3f}{r['per_class_f1']['ET']:>8.3f}{r['pd_vs_et_acc']:>8.2f}")
print("\n-> magnitude WINS: it is rotation-invariant and keeps amplitude-modulation")
print("   structure. The rectification is a NAMING issue (envelope, not tremor), not a bug.")

## 3. PADS cross-dataset

Needs `pads_stretchhold/` from `python -m pdetn.extract_pads`.
Everything below is single-sensor (local **lower_arm** ≈ PADS **wrist**).

In [ ]:
import os
HAVE_PADS = os.path.isdir(PADS_DIR)
if not HAVE_PADS:
    print(f"'{PADS_DIR}' not found - skip sections 3+ (see pdetn/README_PADS.md)")
else:
    pads = load_pads_extracted(PADS_DIR)
    print(f"local {len(local)} recordings | PADS {len(pads)} recordings")
    P = distribution_table(pads, "PADS (StretchHold, wrist)")
    print("\nPD-vs-ET overlap (PADS):",
          {m: round(overlap(P['PD'][m], P['ET'][m]),2) for m in ['peak','mean','med']})
    print("-> the same ~2/3 overlap replicates in an independent cohort.")

### 3a. Build features (~2 min: 832 PADS recordings)

In [ ]:
if HAVE_PADS:
    Xl, yl, sl, _ = build_features(local)
    Xp, yp, sp, _ = build_features(pads)
    print("local", Xl.shape, " PADS", Xp.shape)

### 3b. P1 — transfer (train one dataset, test the other)

In [ ]:
if HAVE_PADS:
    m = TwoStageClassifier("logreg","logreg",tune_et_threshold=True).fit(Xl, yl)
    r = pd_vs_et_metrics(yp, m.predict(Xp))
    print(f"train LOCAL -> test PADS : macroF1={r['macro_f1']:.3f} ET={r['per_class_f1']['ET']:.3f}")
    m2 = TwoStageClassifier("logreg","logreg",tune_et_threshold=True).fit(Xp, yp)
    r2 = pd_vs_et_metrics(yl, m2.predict(Xl))
    print(f"train PADS  -> test LOCAL: macroF1={r2['macro_f1']:.3f} ET={r2['per_class_f1']['ET']:.3f}")
    print("\n-> transfer FAILS both ways (device domain shift).")

### 3c. Domain-shift probe — can a classifier tell the datasets apart?

In [ ]:
if HAVE_PADS:
    Xa, ya, sa, da = build_features(local + pads)
    auc = dataset_identity_probe(Xa, da)
    print(f"dataset-identity AUC = {auc:.3f}   (>0.85 = strong domain shift)")
    # per-dataset standardization = simple domain alignment
    Xal = Xa.copy()
    for d in np.unique(da):
        msk = da==d
        Xal[msk] = (Xa[msk]-np.nanmean(Xa[msk],0))/(np.nanstd(Xa[msk],0)+1e-9)
    print(f"after per-dataset standardization: AUC = {dataset_identity_probe(Xal,da):.3f}")

### 3d. P2 — pooled, and PADS-only (the decisive test)

In [ ]:
if HAVE_PADS:
    def grouped_loso(X, y, subj, n_splits=5):
        pred = np.zeros(len(y), int)
        for tr, te in GroupKFold(n_splits).split(X, y, groups=subj):
            pred[te] = TwoStageClassifier("logreg","logreg",tune_et_threshold=True)\
                       .fit(X[tr], y[tr]).predict(X[te])
        return pred

    for tag, (X,yy,ss) in [("pooled (naive)", (Xa,ya,sa)),
                           ("pooled + alignment", (Xal,ya,sa)),
                           ("PADS-only (41 ET)", (Xp,yp,sp))]:
        pr = grouped_loso(X,yy,ss); r = pd_vs_et_metrics(yy,pr)
        ci = bootstrap_subject_ci(yy,pr,ss,CLASS_NAMES,n_boot=800)['ET']
        print(f"{tag:>20}: macroF1={r['macro_f1']:.3f}  ET={r['per_class_f1']['ET']:.3f} "
              f"[{ci.lo:.2f},{ci.hi:.2f}]  width={ci.hi-ci.lo:.2f}")
    print("\n-> PADS-only has 2.5x the ET subjects: the CI TIGHTENS but ET-F1 does NOT rise.")
    print("   => the low ET-F1 is INTRINSIC, not a small-cohort artifact.")

### 3e. Does adding PADS to training help *your* patients?

In [ ]:
if HAVE_PADS:
    z = lambda X: (X-np.nanmean(X,0))/(np.nanstd(X,0)+1e-9)
    Xln, Xpn = z(Xl), z(Xp)
    for tag, augment in [("your Data only (baseline)", False), ("your Data + PADS", True)]:
        pred = np.zeros(len(yl), int)
        for tr, te in GroupKFold(5).split(Xln, yl, groups=sl):
            Xtr = np.vstack([Xln[tr], Xpn]) if augment else Xln[tr]
            ytr = np.concatenate([yl[tr], yp]) if augment else yl[tr]
            pred[te] = TwoStageClassifier("logreg","logreg",tune_et_threshold=True)\
                       .fit(Xtr,ytr).predict(Xln[te])
        r = pd_vs_et_metrics(yl,pred)
        print(f"{tag:>28}: macroF1={r['macro_f1']:.3f}  ET={r['per_class_f1']['ET']:.3f}")
    print("\n-> adding PADS HURTS. Use PADS as an independent validation cohort, not training data.")

## 4. Summary of findings

| finding | evidence |
|---|---|
| **PD & ET overlap in frequency** | overlap ≈0.5–0.7 on max/mean/median freq in **both datasets**; PD-vs-ET not significant (p≈0.25–0.47); identical median peak (6.64 Hz) and near-identical mean (7.26 vs 7.21 Hz) on lower_arm |
| **N separates cleanly** | Kruskal-Wallis p = 1e-10 … 1e-16 |
| **No TF method helps** | classes aren't separated in frequency → finer resolution has nothing to resolve |
| **Spatial features do help** | ET-F1 0.378 → 0.421 (information orthogonal to frequency) |
| **Keep the magnitude reduction** | rotation-invariant + keeps AM; PC1 is worse (0.421 → 0.378) |
| **PADS can't be combined** | transfer fails, pooling ~0.31, augmentation hurts (0.43 → 0.35); identity AUC 0.999 |
| **ET difficulty is intrinsic** | PADS-only with 41 ET: ET-F1 0.26, tight CI |

**Best model:** local data, `lower_arm` + `OUT`, TF+spatial features, two-stage
logistic regression with in-CV ET threshold tuning.
See `pdetn/two_stage_comparison.ipynb` and the `reports/` folder.